In [ ]:
from omegaconf import OmegaConf
from jupyterscad import view

from solid2 import square, cube

from itertools import accumulate

from configuration import schema, ConfigSchema
from connectors import generate_male_connector, generate_female_connector
from common import generate_keys_row

In [ ]:
yaml_config = OmegaConf.load("default.conf.yaml")
conf = OmegaConf.merge(schema, yaml_config)
conf

In [ ]:
def generate_octave(white_keys: int, conf: ConfigSchema):
    assert white_keys <= 7
    white_keys_to_black_num_mapping = {
        1: 0,
        2: 1,
        3: 2,
        4: 2,
        5: 3,
        6: 4,
        7: 5,
    }

    wk_total_width = conf.white_key_dims.width * conf.dist_u
    octave_width = wk_total_width * white_keys
    white_plate_len = conf.white_key_dims.length * conf.dist_u
    w_distances = [(wk_total_width - conf.mount_u) / 2] + [wk_total_width] * 7
    white_mount_plate = (
        generate_keys_row(
            octave_width,
            white_plate_len,
            w_distances[:white_keys],
            (white_plate_len - conf.mount_u) / 2,
            conf,
        )
        + cube([octave_width, conf.mount_plate_width, conf.base_height_mm]).down(
            conf.base_height_mm
        )
        + generate_female_connector(w_distances[0], white_plate_len, conf)
    )

    bw_diff = conf.white_black_keys_offset_mm + conf.mount_plate_width
    b_distances = [
        wk_total_width - conf.mount_u / 2,
        wk_total_width,
        wk_total_width * 2,
        wk_total_width,
        wk_total_width,
    ]
    black_mount_plate = (
        generate_keys_row(
            octave_width,
            conf.dist_u,
            b_distances[: white_keys_to_black_num_mapping[white_keys]],
            (conf.dist_u - conf.mount_u) / 2,
            conf,
        )
        + cube([octave_width, conf.mount_plate_width, bw_diff]).down(bw_diff)
        + cube([octave_width, conf.mount_plate_width, bw_diff + conf.base_height_mm])
        .down(bw_diff + conf.base_height_mm)
        .translateY(conf.dist_u - conf.mount_plate_width)
        + generate_female_connector(b_distances[0], conf.dist_u, conf)
    )

    return black_mount_plate + white_mount_plate.translate(
        [
            0,
            -white_plate_len,
            -conf.white_black_keys_offset_mm - conf.mount_plate_width,
        ]
    )


view(generate_octave(2, conf))

In [ ]:
# TODO: each octave is indivisible part
#       but it is possible to generate half of the octave
#       currently I have 35 switches, so the max I can get is 2.5 octaves.
#       30~ keys + 5 mods (octave up, octave down, maybe some play, record, etc)

